# 📓 Notebook 02 — EDA & Feature Engineering
## Student Placement Prediction System
### Author: Uttkarsh — EDA, Visualisations & Feature Engineering

This notebook demonstrates:
- Complete Exploratory Data Analysis with interpretations
- Placement class distribution
- Numerical feature distributions by placement class
- Box plots for outlier visualisation
- Correlation heatmap
- Placement rate by key features
- Feature engineering: academic_score and skills_index
- Justification and limitations of derived features

> Run Notebook 01 first, or run `run_full_preprocessing()` in the first cell.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from data_preprocessing import run_full_preprocessing

(X_train, X_test, y_train, y_test,
 preprocessor, df_clean,
 numeric_features, categorical_features, target_col) = run_full_preprocessing()

print(f'\nDataset ready: {df_clean.shape} | Target: {target_col}')

## EDA Part 1: Class Distribution

**Interpretation**: If one class dominates (e.g., 80% placed), accuracy is not a good metric — use F1-score instead.

In [ ]:
counts = df_clean[target_col].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(['Not Placed (0)', 'Placed (1)'],
             [counts.get(0, 0), counts.get(1, 0)],
             color=['#E74C3C', '#2ECC71'], edgecolor='white')
axes[0].set_title('Placement Count')
axes[0].set_ylabel('Students')
for i, v in enumerate([counts.get(0,0), counts.get(1,0)]):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie([counts.get(0,0), counts.get(1,0)],
             labels=['Not Placed', 'Placed'],
             colors=['#E74C3C', '#2ECC71'],
             autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Balance')

plt.suptitle('Placement Class Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

imbalance_ratio = counts.max() / counts.min()
print(f'Imbalance ratio: {imbalance_ratio:.2f}x')
if imbalance_ratio > 1.5:
    print('⚠️  Moderate imbalance detected — use F1-score as primary metric.')
else:
    print('✓ Classes are roughly balanced.')

## EDA Part 2: Numerical Feature Distributions

**Interpretation**: Features where the two class distributions (red vs green) are well-separated are likely stronger predictors. Overlapping distributions suggest less discriminatory power.

In [ ]:
cols_to_plot = numeric_features[:12]  # cap at 12
n = len(cols_to_plot)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).flatten()

palette = {0: '#E74C3C', 1: '#2ECC71'}
for i, col in enumerate(cols_to_plot):
    for cls in [0, 1]:
        data = df_clean[df_clean[target_col] == cls][col].dropna()
        axes[i].hist(data, bins=30, alpha=0.55, color=palette[cls],
                     label=['Not Placed', 'Placed'][cls], density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)
    axes[i].spines[['top','right']].set_visible(False)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numerical Feature Distributions by Placement Class', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## EDA Part 3: Box Plots — Outlier Detection

**Interpretation**: Points beyond the whiskers (1.5×IQR) are potential outliers. Boxes that don't overlap between classes indicate good discriminatory power for that feature.

In [ ]:
cols_to_plot = numeric_features[:9]
n = len(cols_to_plot)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).flatten()

for i, col in enumerate(cols_to_plot):
    data = [df_clean[df_clean[target_col] == 0][col].dropna().values,
            df_clean[df_clean[target_col] == 1][col].dropna().values]
    bp = axes[i].boxplot(data, patch_artist=True,
                          labels=['Not Placed', 'Placed'],
                          medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], ['#E74C3C', '#2ECC71']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[i].set_title(col)
    axes[i].spines[['top', 'right']].set_visible(False)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots: Features vs Placement Class', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## EDA Part 4: Correlation Heatmap

**Interpretation**: 
- Features with high |correlation| with the target (last row/column) are potentially strong predictors.
- Features that are highly correlated with each other may cause multicollinearity in Logistic Regression.
- Correlation does NOT imply causation.

In [ ]:
cols_for_corr = numeric_features + [target_col]
corr_matrix = df_clean[cols_for_corr].corr()

fig, ax = plt.subplots(figsize=(max(8, len(cols_for_corr)), max(6, len(cols_for_corr)-2)))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 9}, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap — Numerical Features + Target', fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr_matrix[target_col].drop(target_col).abs().sort_values(ascending=False)
print('\nTop feature correlations with target (|Pearson|):')
print(target_corr.to_string())

## Feature Engineering

### Feature 1: academic_score
Combines CGPA (longer track record, weight=0.6) and Prev_Sem_Result (recent momentum, weight=0.4). Both normalised to [0,1] before blending.

**Limitation**: The 0.6/0.4 split is a domain-informed heuristic. The actual optimal weights may differ.

In [ ]:
from feature_engineering import apply_feature_engineering

df_fe = apply_feature_engineering(df_clean)
print('New columns added:', [c for c in df_fe.columns if c not in df_clean.columns])

In [ ]:
# Visualise derived features vs placement
derived_cols = [c for c in ['academic_score', 'skills_index'] if c in df_fe.columns]

if derived_cols:
    fig, axes = plt.subplots(1, len(derived_cols), figsize=(6 * len(derived_cols), 4))
    if len(derived_cols) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, derived_cols):
        for cls, color, label in [(0, '#E74C3C', 'Not Placed'), (1, '#2ECC71', 'Placed')]:
            subset = df_fe[df_fe[target_col] == cls][col].dropna()
            ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)
        ax.set_title(f'Distribution of {col}')
        ax.set_xlabel(col)
        ax.legend()
        ax.spines[['top', 'right']].set_visible(False)
    
    plt.suptitle('Derived Features vs Placement Class', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No derived features found — check that source columns exist in your dataset.')

## Summary: EDA Key Findings

| Finding | Implication |
|---------|-------------|
| Class balance (check ratio above) | If imbalanced, prefer F1-score over Accuracy |
| Top correlated features | These are likely the most important predictors |
| Outliers detected | Reported only; tree models handle naturally |
| Derived features | May simplify model or add noise — empirical verification required |